# Amazon Reviews — Sentiment Analysis

**Pipeline:** Data Loading -> EDA -> Text Preprocessing -> Feature Extraction -> Model Training -> Evaluation & Comparison

In [ ]:
# ============================================================
# COLAB SETUP — Chạy cell này đầu tiên khi dùng Google Colab
# ============================================================
import os, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/Duyduide/sentiment-analysis-ml.git"
    REPO_DIR = "/content/sentiment-analysis-ml"

    # 1. Clone hoặc pull repo
    if not os.path.exists(REPO_DIR):
        os.system(f"git clone {REPO_URL} {REPO_DIR}")
    else:
        os.system(f"git -C {REPO_DIR} pull")

    # 2. Chuyển working dir vào notebooks/ để các path tương đối (../data/, ../features/) hoạt động đúng
    os.chdir(f"{REPO_DIR}/notebooks")
    sys.path.insert(0, REPO_DIR)

    # 3. Cài thêm thư viện chưa có sẵn trong Colab
    os.system("pip install -q 'kagglehub[pandas-datasets]' wordcloud")

    print("Colab setup xong. Working dir:", os.getcwd())
    print("Python path includes:", REPO_DIR)
else:
    print("Chạy local — bỏ qua Colab setup.")


---
## 1. Set up môi trường

In [ ]:
import sys
import warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import nltk

nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords
STOP_WORDS = set(stopwords.words("english"))

%matplotlib inline
print("Environment ready.")

---
## 2. Load dữ liệu

### Kaggle API Setup (bắt buộc)

**Bước 1 — Tạo token**
1. Vào: https://www.kaggle.com/settings/account
2. Tìm mục **API Tokens (Recommended)** -> nhấn **Generate New Token**
3. Copy token vừa hiện ra

**Bước 2 — Khi notebook yêu cầu**
- Nhập **Kaggle username** và dán **API token** vừa copy

In [ ]:
import kagglehub
import os
import shutil

path = kagglehub.dataset_download("dongrelaxman/amazon-reviews-dataset")
print("Dataset path:", path)

for file in os.listdir(path):
    if file.endswith(".csv"):
        dest = "../data/amazon_reviews.csv"
        os.makedirs("../data", exist_ok=True)
        shutil.copy(os.path.join(path, file), dest)
        print(f"Copied -> {dest}")
        break

In [ ]:
df = pd.read_csv(
    "../data/amazon_reviews.csv",
    engine="python",
    encoding="utf-8",
    on_bad_lines="skip",
)
print(f"Raw shape: {df.shape}")
df.head()

In [ ]:
from modules.preprocessing import extract_rating, convert_sentiment

df["rating_num"] = df["Rating"].apply(extract_rating)
df["sentiment"]  = df["rating_num"].apply(convert_sentiment)

df = df.dropna(subset=["rating_num", "sentiment"]).reset_index(drop=True)
print(f"Shape after label extraction: {df.shape}")
df[["Rating", "rating_num", "sentiment"]].head()

---
## 3. Exploratory Data Analysis

In [ ]:
print("Shape:", df.shape)
print("\nColumn dtypes:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())

In [ ]:
counts = df["sentiment"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(x="sentiment", data=df, order=counts.index, ax=axes[0])
axes[0].set_title("Sentiment Distribution")
axes[0].set_xlabel("Sentiment")
axes[0].set_ylabel("Count")

counts.plot(kind="pie", autopct="%1.1f%%", ax=axes[1])
axes[1].set_title("Sentiment Proportion")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

print("Sentiment counts:")
print(counts)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top10 = df["Country"].value_counts().head(10)
sns.barplot(x=top10.index, y=top10.values, ax=axes[0])
axes[0].set_title("Top 10 Countries by Reviews")
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha="right")

top5 = df["Country"].value_counts().head(5).index
df_top5 = df[df["Country"].isin(top5)]
sns.countplot(x="Country", hue="sentiment", data=df_top5, ax=axes[1])
axes[1].set_title("Sentiment by Top 5 Countries")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha="right")

plt.tight_layout()
plt.show()

In [ ]:
df["text_length"] = df["Review Text"].astype(str).apply(lambda x: len(x.split()))
upper = np.percentile(df["text_length"], 98)

plt.figure(figsize=(8, 4))
sns.histplot(df[df["text_length"] <= upper]["text_length"], bins=50, kde=True)
plt.title("Review Text Length Distribution (98th percentile cap)")
plt.xlabel("Word Count")
plt.show()

print(f"Mean length : {df['text_length'].mean():.1f} words")
print(f"Median length: {df['text_length'].median():.0f} words")

In [ ]:
from modules.preprocessing import preprocess_pipeline

sample_text = " ".join(
    df["Review Text"].sample(5000, random_state=42).astype(str).apply(preprocess_pipeline)
)

wordcloud = WordCloud(
    width=800, height=400,
    stopwords=STOP_WORDS,
    background_color="white",
    max_words=200,
).generate(sample_text)

plt.figure(figsize=(12, 5))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("Most Frequent Words in Reviews", fontsize=14)
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
sns.histplot(df["Review Count"], bins=30, log_scale=True)
plt.title("Reviewer Activity Distribution (log scale)")
plt.xlabel("Review Count")
plt.show()

---
## 4. Text Preprocessing

Áp dụng pipeline: lowercase -> remove URLs/HTML -> remove special chars -> remove stopwords.

In [ ]:
from modules.preprocessing import preprocess_dataframe, download_nltk_resources

download_nltk_resources()

df = preprocess_dataframe(
    df,
    text_col="Review Text",
    output_col="clean_text",
    use_lemmatization=False,
)
df = df[df["clean_text"].str.strip() != ""].reset_index(drop=True)

print(f"Rows after filtering empty texts: {len(df)}")
df[["Review Text", "clean_text"]].head()

---
## 5. Feature Extraction

### 5.1 TF-IDF Features

In [ ]:
from modules.features import FeatureExtractor
from modules.utils import save_features

X_text = df["clean_text"].tolist()
y      = df["sentiment"]

extractor = FeatureExtractor(max_tfidf_features=5000)
X_tfidf   = extractor.extract_tfidf(X_text, fit=True)

print("TF-IDF matrix shape:", X_tfidf.shape)
print("Sample features:", extractor.tfidf_vectorizer.get_feature_names_out()[:20])

save_features(X_tfidf.toarray(), "../features/tfidf_features.npy")

### 5.2 DistilBERT Embeddings

DistilBERT được dùng để trích xuất embeddings từ `[CLS]` token.  
Do chi phí tính toán, chỉ lấy mẫu **5,000** reviews.

In [ ]:
BERT_SAMPLE = 5000
df_bert = df.sample(BERT_SAMPLE, random_state=42).reset_index(drop=True)

X_bert = extractor.extract_bert(df_bert["clean_text"].tolist(), batch_size=32)
y_bert = df_bert["sentiment"]

print("BERT embeddings shape:", X_bert.shape)

save_features(X_bert, "../features/bert_embeddings.npy")

---
## 6. Model Training

### 6.1 TF-IDF + Classical Classifiers

Các mô hình: Logistic Regression, Naive Bayes, Decision Tree.

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from modules.models import get_classifiers, train_model
import numpy as np

le        = LabelEncoder()
y_encoded = le.fit_transform(y)
LABELS    = list(le.classes_)
print("Classes:", LABELS)

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded,
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# Compute per-sample weights to correct class imbalance
cw = compute_class_weight("balanced", classes=np.arange(len(LABELS)), y=y_train)
sw = np.array([cw[label] for label in y_train])
print("Class weights:", dict(zip(LABELS, cw.round(3))))

classifiers = get_classifiers()
# LR and DT use class_weight="balanced" internally (set in get_classifiers)
lr_model    = train_model(classifiers["Logistic Regression"], X_train, y_train)
# NB does not support class_weight -> pass sample_weight explicitly
nb_model    = train_model(classifiers["Naive Bayes"],         X_train, y_train, sample_weight=sw)
dt_model    = train_model(classifiers["Decision Tree"],       X_train, y_train)

print("TF-IDF models trained.")


### 6.2 DistilBERT Embeddings + Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

le_bert        = LabelEncoder()
y_bert_encoded = le_bert.fit_transform(y_bert)

Xb_train, Xb_test, yb_train, yb_test = train_test_split(
    X_bert, y_bert_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_bert_encoded,
)
print(f"Train: {Xb_train.shape}, Test: {Xb_test.shape}")

# class_weight="balanced" accounts for imbalance in the 5k BERT sample
bert_lr_model = train_model(LogisticRegression(max_iter=1000, class_weight="balanced"), Xb_train, yb_train)
print("BERT + Logistic Regression trained.")


---
## 7. Evaluation & Comparison

### 7.1 TF-IDF Model Evaluations

In [ ]:
from modules.models import evaluate_model

y_pred_lr = lr_model.predict(X_test)
y_pred_nb = nb_model.predict(X_test)
y_pred_dt = dt_model.predict(X_test)

acc_lr = evaluate_model(y_test, y_pred_lr, "Logistic Regression", labels=LABELS)
acc_nb = evaluate_model(y_test, y_pred_nb, "Naive Bayes",         labels=LABELS)
acc_dt = evaluate_model(y_test, y_pred_dt, "Decision Tree",       labels=LABELS)

### 7.2 DistilBERT + LR Evaluation

In [ ]:
LABELS_BERT = list(le_bert.classes_)
yb_pred     = bert_lr_model.predict(Xb_test)
acc_bert    = evaluate_model(yb_test, yb_pred, "BERT + Logistic Regression", labels=LABELS_BERT)

### 7.3 Overall Model Comparison

In [ ]:
from modules.models import compare_models

all_results = {
    "LR (TF-IDF)"      : acc_lr,
    "Naive Bayes (TF-IDF)": acc_nb,
    "Decision Tree (TF-IDF)": acc_dt,
    "LR (BERT)"        : acc_bert,
}

comparison_df = compare_models(all_results, title="TF-IDF vs DistilBERT — Model Comparison")
comparison_df